# Autonomous Trailer: a Knowledge-Based Agent
**ICS 3202 Artificial Intelligence | Practical Exercise 4**

**Group members**

| Name | Admission No. |
|------|---------------|
| Eeshah Sabir Khan | 192089 |
| Abdulle Abdullahi Faduma | 191648 |

Repository: `KnowledgeEngineering`

## 1. Introduction

In Practical Exercise 3 we designed a searching agent for a self-driving trailer carrying cargo across Kenya's highway network, for example from Nairobi to Kisumu. We described it with PEAS, abstracted the highways into a directed graph (towns are nodes, highway links are edges weighted by distance in km) and used Uniform Cost Search (UCS) to find the cheapest route. For Nairobi to Kisumu that gave Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu at 370 km.

That agent only knew distances. In Exercise 3 we listed things such as weather, road conditions, traffic and cargo weight as part of the environment, but the search ignored them. In this exercise we use them. The trailer becomes a **knowledge-based agent** with three parts:

1. **Knowledge base (KB):** the graph and distances from Exercise 3, plus facts about the roads and the trip (weather, cargo weight, time of day, fuel).
2. **Inference engine:** if-then rules applied by forward chaining. They derive new facts, such as "this road is blocked for our load".
3. **Search:** the same UCS as Exercise 3, but its edge costs come from what the KB has inferred.

The agent can also explain why it picked a route.

## 2. Groupwork evidence

![Team photo](image-1.png)

## 3. Building the knowledge base

In [16]:
import heapq

Facts are tuples such as `("road", "Nairobi", "Naivasha")`. In rules, variables start with `?`.

In [17]:
class KnowledgeBase:
    def __init__(self):
        self.facts = set()
        self.why = {}          # derived fact -> name of the rule that produced it

    def tell(self, fact):
        self.facts.add(fact)

    def retract(self, predicate):
        # remove every fact that starts with this predicate
        self.facts = {f for f in self.facts if f[0] != predicate}

    def ask(self, *pattern):
        results = []
        for f in self.facts:
            b = match(pattern, f, {})
            if b is not None:
                results.append(b)
        return results

    def holds(self, fact):
        return fact in self.facts


def is_var(x):
    return isinstance(x, str) and x.startswith("?")


def match(pattern, fact, binding):
    if len(pattern) != len(fact):
        return None
    b = dict(binding)
    for p, f in zip(pattern, fact):
        if is_var(p):
            if p in b and b[p] != f:
                return None
            b[p] = f
        elif p != f:
            return None
    return b

### The map from Exercise 3
Towns and distances (km) are exactly the ones from our Exercise 3 graph. Roads work in both directions, so each one is stored twice.

The last three columns are **new knowledge** that Exercise 3 abstracted away: the heaviest load the road allows (tonnes), how busy it usually is, and whether it is steep or landslide-prone in the rain. These values are our own estimates for this exercise, not official figures.

In [18]:
kb = KnowledgeBase()

# (town A, town B, km, load limit in tonnes, usual traffic, steep/slippery when wet?)
roads = [
    ("Nairobi",   "Thika",     45,  40, "high",   False),
    ("Thika",     "Nyeri",     120, 40, "medium", False),
    ("Nyeri",     "Karatina",  70,  40, "low",    False),
    ("Karatina",  "Nanyuki",   70,  40, "low",    False),
    ("Nairobi",   "Naivasha",  89,  40, "high",   True),    # escarpment section
    ("Naivasha",  "Nakuru",    76,  40, "high",   False),
    ("Nakuru",    "Kericho",   105, 35, "medium", True),
    ("Kericho",   "Kisumu",    100, 40, "medium", False),
    ("Nakuru",    "Eldoret",   160, 40, "high",   False),
    ("Eldoret",   "Kakamega",  100, 40, "medium", False),
    ("Kakamega",  "Kisumu",    50,  40, "medium", False),
    ("Nairobi",   "Narok",     140, 40, "medium", True),
    ("Narok",     "Bomet",     100, 30, "low",    False),
    ("Bomet",     "Kericho",   50,  30, "low",    False),
]

for a, b, km, limit, traffic, steep in roads:
    for x, y in ((a, b), (b, a)):
        kb.tell(("road", x, y))
        kb.tell(("distance", x, y, km))
        kb.tell(("load_limit", x, y, limit))
        kb.tell(("traffic", x, y, traffic))
        if steep:
            kb.tell(("steep", x, y))

# towns where the trailer can refuel (assumed: every town in our graph has a fuel station)
towns = sorted({r[0] for r in roads} | {r[1] for r in roads})
for t in towns:
    kb.tell(("fuel_station", t))

print(len(towns), "towns,", len(kb.facts), "facts in the knowledge base")

13 towns, 131 facts in the knowledge base


## 4. Rules and the inference engine
The current conditions (weather, cargo weight, time of day, fuel) are also facts. The rules combine them with the permanent facts above. A condition starting with `<` or `>` is a numeric comparison.

In [19]:
rules = [
    ("R1: cargo is heavier than the road allows",
     [("load_limit", "?a", "?b", "?lim"), ("cargo_weight", "?w"), ("<", "?lim", "?w")],
     ("blocked", "?a", "?b")),

    ("R2: steep road in the rain is dangerous",
     [("steep", "?a", "?b"), ("weather", "rainy")],
     ("avoid", "?a", "?b")),

    ("R3: busy road at peak hours means delays",
     [("traffic", "?a", "?b", "high"), ("time_of_day", "peak")],
     ("slow", "?a", "?b")),

    ("R4: heavy cargo on a steep road is risky even when dry",
     [("steep", "?a", "?b"), ("cargo_weight", "?w"), (">", "?w", 30)],
     ("avoid", "?a", "?b")),
]


def satisfy(conditions, kb, binding):
    if not conditions:
        yield binding
        return
    first, rest = conditions[0], conditions[1:]
    if first[0] in ("<", ">"):
        left = binding.get(first[1], first[1])
        right = binding.get(first[2], first[2])
        ok = left < right if first[0] == "<" else left > right
        if ok:
            yield from satisfy(rest, kb, binding)
        return
    for fact in list(kb.facts):
        b = match(first, fact, binding)
        if b is not None:
            yield from satisfy(rest, kb, b)


def fill(pattern, binding):
    return tuple(binding.get(p, p) for p in pattern)


def forward_chain(kb):
    # clear old conclusions, then apply the rules until nothing new appears
    for pred in ("blocked", "avoid", "slow"):
        kb.retract(pred)
    kb.why = {}
    changed = True
    while changed:
        changed = False
        for name, conds, concl in rules:
            for b in satisfy(conds, kb, {}):
                new_fact = fill(concl, b)
                if new_fact not in kb.facts:
                    kb.facts.add(new_fact)
                    kb.why[new_fact] = name
                    changed = True

## 5. Search using the knowledge
This is the Uniform Cost Search from Exercise 3: always expand the frontier path with the lowest accumulated cost, and stop when the goal is taken off the frontier. The difference is the edge cost. A blocked road is not used at all, a road we should avoid costs 3 times its distance, and a slow road costs 1.5 times. The multipliers are our own choice.

In [20]:
def road_cost(kb, a, b):
    if kb.holds(("blocked", a, b)):
        return None
    km = kb.ask("distance", a, b, "?km")[0]["?km"]
    if kb.holds(("avoid", a, b)):
        return km * 3
    if kb.holds(("slow", a, b)):
        return km * 1.5
    return km


def ucs(kb, start, goal, show_frontier=False):
    frontier = [(0, start, [start])]
    explored = set()
    step = 0
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in explored:
            continue
        step += 1
        if show_frontier:
            print(step, "expand", node, "(" + str(cost) + ")")
        if node == goal:
            return path, cost
        explored.add(node)
        for nb in kb.ask("road", node, "?next"):
            nxt = nb["?next"]
            c = road_cost(kb, node, nxt)
            if c is None or nxt in explored:
                continue
            heapq.heappush(frontier, (cost + c, nxt, path + [nxt]))
    return None, None

## 6. The agent

In [21]:
FUEL_PER_KM = 0.35      # litres per km for a loaded trailer (assumed value)


class TrailerAgent:
    def __init__(self, kb, location="Nairobi"):
        self.kb = kb
        self.location = location

    def sense(self, weather, cargo, time_of_day, fuel):
        # update the KB with what the sensors report, then re-run the rules
        for p in ("weather", "cargo_weight", "time_of_day", "fuel_litres"):
            self.kb.retract(p)
        self.kb.tell(("weather", weather))
        self.kb.tell(("cargo_weight", cargo))
        self.kb.tell(("time_of_day", time_of_day))
        self.kb.tell(("fuel_litres", fuel))
        forward_chain(self.kb)

    def plan(self, goal):
        path, cost = ucs(self.kb, self.location, goal)
        if path is None:
            return None
        km = sum(self.kb.ask("distance", a, b, "?km")[0]["?km"] for a, b in zip(path, path[1:]))
        return {"path": path, "cost": cost, "km": km}

    def explain(self, plan):
        lines = []
        for a, b in zip(plan["path"], plan["path"][1:]):
            note = "normal"
            for tag in ("avoid", "slow"):
                if (tag, a, b) in self.kb.why:
                    note = self.kb.why[(tag, a, b)]
            lines.append("  " + a + " -> " + b + "  (" + note + ")")
        seen = set()
        for f, rule in self.kb.why.items():
            if f[0] == "blocked":
                key = tuple(sorted((f[1], f[2])))
                if key not in seen:
                    seen.add(key)
                    lines.append("  [not used] " + key[0] + " - " + key[1] + ": " + rule)
        return "\n".join(lines)

    def fuel_check(self, plan):
        fuel = self.kb.ask("fuel_litres", "?l")[0]["?l"]
        needed = plan["km"] * FUEL_PER_KM
        if needed <= fuel:
            return "Fuel ok: need about %.0f L, have %.0f L." % (needed, fuel)
        # find the last town on the route where we can still reach a fuel station
        msg = "Not enough fuel: need about %.0f L, have %.0f L." % (needed, fuel)
        reach = 0
        stop = None
        for a, b in zip(plan["path"], plan["path"][1:]):
            reach += self.kb.ask("distance", a, b, "?km")[0]["?km"] * FUEL_PER_KM
            if reach <= fuel and self.kb.holds(("fuel_station", b)):
                stop = b
        if stop:
            msg += " Refuel at " + stop + " on the way."
        else:
            msg += " Refuel before leaving."
        return msg

    def go(self, goal):
        plan = self.plan(goal)
        if plan is None:
            print("No safe route from", self.location, "to", goal, "under current conditions.")
            return
        print("Route:", " -> ".join(plan["path"]))
        print("Distance: %d km (effective cost %g)" % (plan["km"], plan["cost"]))
        print(self.fuel_check(plan))
        print("Reasoning:")
        print(self.explain(plan))
        self.location = goal

## 7. Testing the agent
Scenario 1 uses normal conditions and should match our Exercise 3 answer (370 km via Naivasha, Nakuru and Kericho). The other scenarios change one thing at a time.

In [22]:
print("=== Scenario 1: dry, light cargo, off-peak (should match Exercise 3) ===")
agent = TrailerAgent(kb, "Nairobi")
agent.sense(weather="dry", cargo=10, time_of_day="offpeak", fuel=200)
agent.go("Kisumu")

=== Scenario 1: dry, light cargo, off-peak (should match Exercise 3) ===
Route: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Distance: 370 km (effective cost 370)
Fuel ok: need about 130 L, have 200 L.
Reasoning:
  Nairobi -> Naivasha  (normal)
  Naivasha -> Nakuru  (normal)
  Nakuru -> Kericho  (normal)
  Kericho -> Kisumu  (normal)


In [23]:
print("=== Scenario 2: raining ===")
agent = TrailerAgent(kb, "Nairobi")
agent.sense(weather="rainy", cargo=10, time_of_day="offpeak", fuel=200)
agent.go("Kisumu")

=== Scenario 2: raining ===
Route: Nairobi -> Naivasha -> Nakuru -> Eldoret -> Kakamega -> Kisumu
Distance: 475 km (effective cost 653)
Fuel ok: need about 166 L, have 200 L.
Reasoning:
  Nairobi -> Naivasha  (R2: steep road in the rain is dangerous)
  Naivasha -> Nakuru  (normal)
  Nakuru -> Eldoret  (normal)
  Eldoret -> Kakamega  (normal)
  Kakamega -> Kisumu  (normal)


In [24]:
print("=== Scenario 3: 36 tonne cargo ===")
agent = TrailerAgent(kb, "Nairobi")
agent.sense(weather="dry", cargo=36, time_of_day="offpeak", fuel=200)
agent.go("Kisumu")

=== Scenario 3: 36 tonne cargo ===
Route: Nairobi -> Naivasha -> Nakuru -> Eldoret -> Kakamega -> Kisumu
Distance: 475 km (effective cost 653)
Fuel ok: need about 166 L, have 200 L.
Reasoning:
  Nairobi -> Naivasha  (R4: heavy cargo on a steep road is risky even when dry)
  Naivasha -> Nakuru  (normal)
  Nakuru -> Eldoret  (normal)
  Eldoret -> Kakamega  (normal)
  Kakamega -> Kisumu  (normal)
  [not used] Kericho - Nakuru: R1: cargo is heavier than the road allows
  [not used] Bomet - Kericho: R1: cargo is heavier than the road allows
  [not used] Bomet - Narok: R1: cargo is heavier than the road allows


In [25]:
print("=== Scenario 4: peak hours, going to Eldoret ===")
agent = TrailerAgent(kb, "Nairobi")
agent.sense(weather="dry", cargo=10, time_of_day="peak", fuel=200)
agent.go("Eldoret")

=== Scenario 4: peak hours, going to Eldoret ===
Route: Nairobi -> Naivasha -> Nakuru -> Eldoret
Distance: 325 km (effective cost 487.5)
Fuel ok: need about 114 L, have 200 L.
Reasoning:
  Nairobi -> Naivasha  (R3: busy road at peak hours means delays)
  Naivasha -> Nakuru  (R3: busy road at peak hours means delays)
  Nakuru -> Eldoret  (R3: busy road at peak hours means delays)


In [26]:
print("=== Scenario 5: low fuel ===")
agent = TrailerAgent(kb, "Nairobi")
agent.sense(weather="dry", cargo=10, time_of_day="offpeak", fuel=60)
agent.go("Kisumu")

=== Scenario 5: low fuel ===
Route: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Distance: 370 km (effective cost 370)
Not enough fuel: need about 130 L, have 60 L. Refuel at Nakuru on the way.
Reasoning:
  Nairobi -> Naivasha  (normal)
  Naivasha -> Nakuru  (normal)
  Nakuru -> Kericho  (normal)
  Kericho -> Kisumu  (normal)


## 8. User interaction
A simple text interface. The user gives the destination and the current conditions and the agent answers. Type `quit` to stop. We used a command-line loop because it works in Jupyter and Colab without extra setup.

In [27]:
def chat():
    agent = TrailerAgent(kb, "Nairobi")
    print("Autonomous Trailer ready. Towns:", ", ".join(towns))
    while True:
        print("\nCurrently at", agent.location)
        goal = input("Destination (or 'quit'): ").strip().title()
        if goal.lower() == "quit":
            break
        if goal not in towns:
            print("I don't know that town.")
            continue
        weather = input("Weather (dry/rainy): ").strip().lower() or "dry"
        cargo = float(input("Cargo weight in tonnes: ") or 10)
        tod = input("Time (peak/offpeak): ").strip().lower() or "offpeak"
        fuel = float(input("Fuel in litres: ") or 200)
        agent.sense(weather, cargo, tod, fuel)
        agent.go(goal)

# chat()   # uncomment to run it interactively

## 9. Using an API for the knowledge base
The exercise encourages APIs where permitted. As an extension, the weather fact can come from Open-Meteo, which is free and needs no key. If there is no internet, it falls back to "dry".

In [28]:
def live_weather(lat=-1.2864, lon=36.8172):
    # current precipitation near Nairobi; returns "rainy" or "dry"
    try:
        import requests
        r = requests.get("https://api.open-meteo.com/v1/forecast",
                         params={"latitude": lat, "longitude": lon, "current": "precipitation"},
                         timeout=5)
        return "rainy" if r.json()["current"]["precipitation"] > 0 else "dry"
    except Exception as e:
        print("Could not get live weather (" + str(e) + "), assuming dry.")
        return "dry"

# agent.sense(weather=live_weather(), cargo=10, time_of_day="offpeak", fuel=200)

## 10. Discussion

**What makes it knowledge-based?** The decisions are not hard-coded into the search. The graph, the road properties and the current conditions are facts, and the rules derive what is blocked, risky or slow. To change the agent we edit facts or rules, and the UCS code stays the same. It also explains its choices using the rule that fired.

**Link to Exercise 3.** In our PEAS description, we said that the environment is made of weather, road conditions, traffic and cargo weight and that the performance measure is to obey rules and to use fuel well. It is the knowledge base in which they finally influence the decision of the agent. The environment remains partially observable, stochastic and dynamic, so in a real trailer the `sense` step would be repeated as conditions change.

**Limitations**
- Load limits, traffic levels and steep sections are our own estimates, not official data.
- Siaya was in our list of towns but had no roads in the Exercise 3 graph, so it is not included here.
- The cost multipliers (3x and 1.5x) and fuel use per km are assumed values.
- The agent senses conditions once, before planning. It does not re-plan during the trip.

**Possible improvements:** live traffic and road-closure data from an API, more towns and roads, re-planning when conditions change, and refuelling stops built into the search itself.

## 10. Discussion

**Why is it knowledge based?** The search is not hard-coded with decisions. The graph, the road properties and the current conditions are facts and the rules determine what is blocked, risky or slow. Change agent by editing facts or rules. UCS code remains the same. It can also explain its choices with the rule that was fired.



**Restrictions**and the scenarios show the route changing when the weather, cargo weight, time of day or fuel change.